In [17]:
# 실습에 필요한 패키지를 설치합니다.
!pip install -r ../requirements.txt

# 03 Multi Step Workflow

대상 workflow
- classify
- route
- search
- answer
- retry

## 왜 classify와 route가 먼저일까?

`classify`는 지금 들어온 질문이 어떤 종류인지 판단하는 단계다.
예를 들어 환불 문의인지, 운영 시간 문의인지, 회사 소개 요청인지 먼저 정리해두면 다음 단계가 단순해진다.

`route`는 분류 결과를 바탕으로 어디로 보낼지 결정하는 단계다.
예를 들어 `refund_policy`로 분류되면 환불 정책 문서를 보고, `contact_info`로 분류되면 연락처 문서를 보게 만들 수 있다.

즉,
- classify = 무엇에 대한 질문인지 정리
- route = 어떤 처리 경로를 탈지 결정

이 두 단계가 잘 되어야 뒤의 `search`, `answer`, `retry`도 안정적으로 동작한다.

In [18]:
# OpenAI 클라이언트와 예제 문서를 준비합니다.
import os
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# 이번 노트북에서 공통으로 사용할 경로와 모델 이름입니다.
DATA_DIR = Path("../data")
MODEL_NAME = "gpt-4.1-mini"

# route 단계에서 참조할 원문 문서를 메모리에 올립니다.
docs = {
    "business_hours": (DATA_DIR / "business_hours.txt").read_text(encoding="utf-8").strip(),
    "refund_policy": (DATA_DIR / "refund_policy.txt").read_text(encoding="utf-8").strip(),
    "contact_info": (DATA_DIR / "contact_info.txt").read_text(encoding="utf-8").strip(),
    "company_intro": (DATA_DIR / "company_intro.txt").read_text(encoding="utf-8").strip(),
}

print("OPENAI_API_KEY loaded:", bool(api_key))
print("사용 가능한 문서:", list(docs.keys()))

OPENAI_API_KEY loaded: True
사용 가능한 문서: ['business_hours', 'refund_policy', 'contact_info', 'company_intro']


In [19]:
# 각 문서가 어떤 내용인지 미리 훑어봅니다.
for name, text in docs.items():
    preview = text[:120].replace("\n", " ")
    print(f"[{name}]")
    print(preview)
    print()

[business_hours]
운영 시간  고객센터 운영 시간은 평일 오전 10시부터 오후 6시까지입니다. 점심시간은 오후 12시 30분부터 오후 1시 30분까지이며 해당 시간에는 전화 상담이 어렵습니다. 토요일, 일요일, 공휴일에는 고객센터를 

[refund_policy]
환불 정책  결제 후 7일 이내이며 강의를 한 번도 수강하지 않은 경우 전액 환불이 가능합니다. 결제 후 7일이 지났거나 강의 영상을 20퍼센트 이상 수강한 경우에는 전액 환불이 어렵습니다. 강의 영상을 20퍼센트 

[contact_info]
연락처  대표 이메일은 support@ailabstore.example 입니다. 대표 전화번호는 02-1234-5678 입니다. 카카오톡 상담 채널 이름은 에이아이 랩 스토어 고객센터입니다. 환불, 결제, 수강 관련

[company_intro]
회사 소개  에이아이 랩 스토어는 직장인과 대학생을 위한 AI 학습 콘텐츠를 제공하는 온라인 교육 플랫폼입니다. 주요 서비스는 실습형 강의, 주간 라이브 세션, 프로젝트 피드백, 학습 자료 다운로드입니다. 본사는 서



## 1. Intent 라벨 설계

먼저 질문을 어떤 카테고리로 나눌지 정해야 한다.
라벨이 너무 많으면 분류가 어려워지고, 너무 적으면 서로 다른 질문이 한데 섞인다.

이번 실습에서는 아래 5개 라벨을 사용한다.
- `business_hours`
- `refund_policy`
- `contact_info`
- `company_intro`
- `other`

`other`는 범위 밖 질문이나 애매한 질문을 안전하게 받기 위한 라벨이다.

In [20]:
# 분류에 사용할 intent 라벨과 라우팅 테이블을 정의합니다.
LABELS = [
    "business_hours",
    "refund_policy",
    "contact_info",
    "company_intro",
    "other",
]

ROUTE_TABLE = {
    "business_hours": {"type": "document", "target": "business_hours"},
    "refund_policy": {"type": "document", "target": "refund_policy"},
    "contact_info": {"type": "document", "target": "contact_info"},
    "company_intro": {"type": "document", "target": "company_intro"},
    "other": {"type": "fallback", "target": None},
}

# 어떤 라벨이 어떤 경로로 가는지 한눈에 확인합니다.
pprint(ROUTE_TABLE)

{'business_hours': {'target': 'business_hours', 'type': 'document'},
 'company_intro': {'target': 'company_intro', 'type': 'document'},
 'contact_info': {'target': 'contact_info', 'type': 'document'},
 'other': {'target': None, 'type': 'fallback'},
 'refund_policy': {'target': 'refund_policy', 'type': 'document'}}


## 2. Rule-based Classify

가장 먼저 해볼 수 있는 방법은 규칙 기반 분류다.
질문 안에 특정 키워드가 있는지 보고 라벨을 붙인다.

장점
- 빠르다
- 동작이 예측 가능하다
- 디버깅이 쉽다

단점
- 표현이 조금만 달라져도 놓칠 수 있다
- 복합 질문 처리에 약하다

In [21]:
# 간단한 키워드 규칙으로 intent를 분류합니다.
def classify_rule_based(query: str) -> str:
    # 대소문자 영향을 줄이기 위해 소문자로 정규화합니다.
    q = query.lower()

    # intent별로 대표 키워드를 정의합니다.
    refund_keywords = ["환불", "취소", "refund"]
    hours_keywords = ["운영", "영업", "몇 시", "시간", "주말"]
    contact_keywords = ["연락처", "전화", "이메일", "메일", "카카오톡", "문의"]
    intro_keywords = ["회사 소개", "무슨 회사", "어떤 회사", "소개", "서비스"]

    # 먼저 매칭되는 intent를 반환합니다.
    if any(keyword in q for keyword in refund_keywords):
        return "refund_policy"
    if any(keyword in q for keyword in hours_keywords):
        return "business_hours"
    if any(keyword in q for keyword in contact_keywords):
        return "contact_info"
    if any(keyword in q for keyword in intro_keywords):
        return "company_intro"
    # 어떤 규칙에도 걸리지 않으면 other로 보냅니다.
    return "other"

In [22]:
# 규칙 기반 분류기가 예상대로 동작하는지 테스트합니다.
test_queries = [
    "환불은 언제까지 가능한가요?",
    "고객센터 몇 시까지 운영해요?",
    "문의 이메일 알려주세요.",
    "어떤 회사인지 소개해주세요.",
    "채용 중인가요?",
]

for query in test_queries:
    label = classify_rule_based(query)
    print(f"query: {query}")
    print(f"label: {label}")
    print()

query: 환불은 언제까지 가능한가요?
label: refund_policy

query: 고객센터 몇 시까지 운영해요?
label: business_hours

query: 문의 이메일 알려주세요.
label: contact_info

query: 어떤 회사인지 소개해주세요.
label: company_intro

query: 채용 중인가요?
label: other



## 3. LLM 기반 Classify

규칙 기반 방식 다음에는 LLM에게 분류를 맡길 수 있다.
이 방식은 표현이 조금 달라져도 더 유연하게 대응할 수 있다.

여기서는 구조화된 출력으로 `label`과 `reason`을 함께 받는다.
이렇게 하면 단순히 결과만 보는 것보다 왜 그렇게 분류했는지도 같이 확인할 수 있다.

In [23]:
# LLM 분류 결과를 구조화해서 받기 위한 스키마를 정의합니다.
from typing import Literal
from pydantic import BaseModel

class ClassificationResult(BaseModel):
    label: Literal[
        "business_hours",
        "refund_policy",
        "contact_info",
        "company_intro",
        "other",
    ]
    reason: str


# OpenAI Responses API를 사용해 intent와 이유를 함께 분류합니다.
def classify_with_llm(query: str) -> ClassificationResult:
    response = client.responses.parse(
        model=MODEL_NAME,
        instructions=(
            "너는 사용자 질문을 intent label로 분류하는 라우팅 전 단계 분류기다. "
            "반드시 주어진 라벨 중 하나만 선택하고, 짧은 이유를 함께 반환하라."
        ),
        input=(
            f"가능한 라벨: {LABELS}\n"
            f"사용자 질문: {query}"
        ),
        text_format=ClassificationResult,
    )
    # parse 결과에서 구조화된 객체만 꺼내 반환합니다.
    return response.output_parsed

In [24]:
# LLM 분류기를 단건으로 실험할 때 사용하는 예시입니다.
# result = classify_with_llm("환불 문의는 어디로 하면 되나요?")
# print(result)

## 4. Route

이제 분류 결과를 바탕으로 처리 경로를 정한다.

`route`는 답변을 직접 만드는 단계가 아니라, 다음에 어떤 처리를 할지 고르는 단계다.

예를 들어
- `refund_policy`면 환불 정책 문서를 참고하는 경로로 보낸다
- `contact_info`면 연락처 문서를 참고하는 경로로 보낸다
- `other`면 fallback 또는 retry 후보 경로로 보낸다

즉, `classify`가 라벨을 붙였다면 `route`는 그 라벨을 실제 실행 경로로 바꾸는 역할을 한다.

여기서는 route 결과를 딕셔너리로 반환해서, 뒤 단계가 그대로 이어받아 사용할 수 있게 만든다.

In [25]:
# 분류 라벨을 실제 처리 경로 정보로 변환합니다.
def route_by_label(label: str) -> dict:
    # 라벨에 대응하는 route 설정을 조회합니다.
    route = ROUTE_TABLE[label]

    # 문서 기반 경로인 경우, 이후 단계가 쓸 수 있게 원문까지 함께 넘깁니다.
    if route["type"] == "document":
        target = route["target"]
        return {
            "type": "document",
            "target": target,
            "description": f"{target} 문서를 참고하는 경로로 보냅니다.",
            "next_step": "search_or_read_document",
            "content": docs[target],
        }

    # 범위 밖 질문은 fallback/retry 경로로 보냅니다.
    return {
        "type": "fallback",
        "target": None,
        "description": "준비된 문서 범위를 벗어나므로 fallback 또는 retry 경로로 보냅니다.",
        "next_step": "fallback_or_retry",
        "content": "현재 준비된 문서 범위를 벗어난 질문입니다.",
    }

In [26]:
# 단일 질문에 대해 route 결과가 어떻게 나오는지 확인합니다.
query = "고객센터 운영 시간이 어떻게 되나요?"
label = classify_rule_based(query)
route_result = route_by_label(label)

print("query:", query)
print("label:", label)
print("route type:", route_result["type"])
print("route target:", route_result["target"])
print("route description:", route_result["description"])
print("next step:", route_result["next_step"])
print()
print(route_result["content"][:300])

query: 고객센터 운영 시간이 어떻게 되나요?
label: business_hours
route type: document
route target: business_hours
route description: business_hours 문서를 참고하는 경로로 보냅니다.
next step: search_or_read_document

운영 시간

고객센터 운영 시간은 평일 오전 10시부터 오후 6시까지입니다.
점심시간은 오후 12시 30분부터 오후 1시 30분까지이며 해당 시간에는 전화 상담이 어렵습니다.
토요일, 일요일, 공휴일에는 고객센터를 운영하지 않습니다.
온라인 강의 시청과 자료 다운로드는 24시간 이용할 수 있습니다.
라이브 세션 문의는 세션 시작 1시간 전까지 접수하는 것을 권장합니다.


In [27]:
# 여러 질문을 한 번에 넣어 route 패턴을 비교합니다.
route_queries = [
    "환불은 며칠 안에 요청해야 하나요?",
    "대표 이메일 주소 알려주세요.",
    "주말에도 고객센터 하나요?",
    "어떤 서비스를 제공하나요?",
    "채용 중인가요?",
]

for query in route_queries:
    label = classify_rule_based(query)
    route_result = route_by_label(label)
    print("=" * 60)
    print("query:", query)
    print("label:", label)
    print("route target:", route_result["target"])
    print("next step:", route_result["next_step"])
    print("description:", route_result["description"])


query: 환불은 며칠 안에 요청해야 하나요?
label: refund_policy
route target: refund_policy
next step: search_or_read_document
description: refund_policy 문서를 참고하는 경로로 보냅니다.
query: 대표 이메일 주소 알려주세요.
label: contact_info
route target: contact_info
next step: search_or_read_document
description: contact_info 문서를 참고하는 경로로 보냅니다.
query: 주말에도 고객센터 하나요?
label: business_hours
route target: business_hours
next step: search_or_read_document
description: business_hours 문서를 참고하는 경로로 보냅니다.
query: 어떤 서비스를 제공하나요?
label: company_intro
route target: company_intro
next step: search_or_read_document
description: company_intro 문서를 참고하는 경로로 보냅니다.
query: 채용 중인가요?
label: other
route target: None
next step: fallback_or_retry
description: 준비된 문서 범위를 벗어나므로 fallback 또는 retry 경로로 보냅니다.


## 5. classify + route 묶기

실제 멀티스텝 workflow에서는 단계를 따로 만들되, 하나의 함수로도 쉽게 엮을 수 있어야 한다.
아래 함수는 지금 단계에서 `classify -> route`까지만 수행한다.

다음 섹션에서는 여기에 `search`를 실제로 붙여본다.

In [28]:
# classify와 route를 한 번에 실행하는 미리보기 함수입니다.
def run_workflow_preview(query: str, use_llm_classifier: bool = False) -> dict:
    # 필요에 따라 규칙 기반 분류기와 LLM 분류기를 선택합니다.
    if use_llm_classifier:
        classification = classify_with_llm(query)
        label = classification.label
        reason = classification.reason
    else:
        label = classify_rule_based(query)
        reason = "rule-based classifier"

    # 분류 결과를 route 단계에 넘깁니다.
    route_result = route_by_label(label)

    # 다음 단계가 바로 사용할 수 있게 중간 상태를 묶어 반환합니다.
    return {
        "query": query,
        "classification": {
            "label": label,
            "reason": reason,
        },
        "route": route_result,
    }

In [29]:
# classify -> route 흐름을 여러 질문으로 확인합니다.
sample_queries = [
    "환불은 며칠 안에 요청해야 하나요?",
    "대표 이메일 주소 알려주세요.",
    "주말에도 고객센터 하나요?",
    "당신들은 어떤 서비스를 제공하나요?",
    "채용 공고는 어디서 보나요?",
]

for query in sample_queries:
    result = run_workflow_preview(query)
    print("=" * 60)
    print("query:", result["query"])
    print("label:", result["classification"]["label"])
    print("reason:", result["classification"]["reason"])
    print("route target:", result["route"]["target"])
    print("route type:", result["route"]["type"])

query: 환불은 며칠 안에 요청해야 하나요?
label: refund_policy
reason: rule-based classifier
route target: refund_policy
route type: document
query: 대표 이메일 주소 알려주세요.
label: contact_info
reason: rule-based classifier
route target: contact_info
route type: document
query: 주말에도 고객센터 하나요?
label: business_hours
reason: rule-based classifier
route target: business_hours
route type: document
query: 당신들은 어떤 서비스를 제공하나요?
label: company_intro
reason: rule-based classifier
route target: company_intro
route type: document
query: 채용 공고는 어디서 보나요?
label: other
reason: rule-based classifier
route target: None
route type: fallback


## 6. Search

`route`가 어느 문서를 볼지 정했다면, `search`는 그 문서 안에서 어떤 내용이 실제로 관련 있는지 좁혀주는 단계다.

즉,
- `route` = 어느 문서나 시스템으로 갈지 결정
- `search` = 그 문서 안에서 어떤 부분을 꺼낼지 결정

이번 노트북에서는 이해를 쉽게 하기 위해 벡터 검색 대신 간단한 키워드 기반 검색을 사용한다.
질문과 각 문장 사이의 단어 겹침 정도를 계산해서 가장 관련 있는 문장을 찾는다.

In [30]:
# 검색을 위해 문서를 문장/행 단위 passage로 나눕니다.
def split_into_passages(text: str) -> list[str]:
    passages = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            passages.append(line)
    return passages


# 아주 단순한 방식으로 문장을 토큰 집합으로 바꿉니다.
def tokenize_korean_text(text: str) -> set[str]:
    cleaned = text.lower()
    for ch in [",", ".", "?", "!", ":", ";", "(", ")"]:
        cleaned = cleaned.replace(ch, " ")
    return {token for token in cleaned.split() if len(token) >= 2}


# route가 고른 문서 안에서 질문과 가장 관련 있는 passage를 찾습니다.
def search_step(query: str, route_result: dict, top_k: int = 2) -> dict:
    if route_result["type"] != "document":
        # fallback 경로면 검색 자체를 수행하지 않습니다.
        return {
            "type": "no_search",
            "target": route_result["target"],
            "matches": [],
            "reason": "fallback 경로이므로 문서 검색을 수행하지 않습니다.",
        }

    # 질문과 문서를 같은 방식으로 토큰화한 뒤 겹치는 단어 수를 점수로 사용합니다.
    query_tokens = tokenize_korean_text(query)
    passages = split_into_passages(route_result["content"])
    scored_matches = []

    for passage in passages:
        passage_tokens = tokenize_korean_text(passage)
        overlap = query_tokens & passage_tokens
        score = len(overlap)
        scored_matches.append(
            {
                "passage": passage,
                "score": score,
                "overlap_tokens": sorted(overlap),
            }
        )

    # 점수가 높은 순으로 정렬해 상위 passage만 반환합니다.
    ranked = sorted(scored_matches, key=lambda item: item["score"], reverse=True)

    return {
        "type": "document_search",
        "target": route_result["target"],
        "matches": ranked[:top_k],
        "reason": f"{route_result['target']} 문서 안에서 질문과 겹치는 표현이 많은 문장을 찾았습니다.",
    }

In [31]:
# 한 질문에 대해 search 결과가 어떤 근거를 뽑는지 확인합니다.
query = "환불은 언제까지 가능하고 처리까지 며칠 걸리나요?"
label = classify_rule_based(query)
route_result = route_by_label(label)
search_result = search_step(query, route_result)

print("query:", query)
print("label:", label)
print("route target:", route_result["target"])
print("search reason:", search_result["reason"])
print()

for idx, match in enumerate(search_result["matches"], start=1):
    print(f"[{idx}] score={match['score']}")
    print("overlap tokens:", match["overlap_tokens"])
    print(match["passage"])
    print()

query: 환불은 언제까지 가능하고 처리까지 며칠 걸리나요?
label: refund_policy
route target: refund_policy
search reason: refund_policy 문서 안에서 질문과 겹치는 표현이 많은 문장을 찾았습니다.

[1] score=1
overlap tokens: ['환불은']
환불은 고객센터 이메일 또는 대표 전화로 접수할 수 있으며 접수 후 영업일 기준 3일 이내 처리됩니다.

[2] score=0
overlap tokens: []
환불 정책



## 7. classify + route + search 묶기

이제 `classify`, `route`, `search`를 하나의 함수로 묶어보면 멀티스텝 workflow의 흐름이 더 잘 보인다.

최종 흐름은 다음과 같다.

`query -> classification -> route -> search`

In [32]:
# search 단계까지 포함한 중간 workflow 상태를 반환합니다.
def run_workflow_until_search(query: str, use_llm_classifier: bool = False) -> dict:
    if use_llm_classifier:
        classification = classify_with_llm(query)
        label = classification.label
        reason = classification.reason
    else:
        label = classify_rule_based(query)
        reason = "rule-based classifier"

    # 분류 결과를 route로 보내고, route 결과를 search로 넘깁니다.
    route_result = route_by_label(label)
    search_result = search_step(query, route_result)

    # 이후 answer/retry에서 그대로 쓸 수 있게 상태를 묶어 반환합니다.
    return {
        "query": query,
        "classification": {
            "label": label,
            "reason": reason,
        },
        "route": route_result,
        "search": search_result,
    }

In [33]:
# classify -> route -> search 흐름을 여러 질문으로 확인합니다.
search_queries = [
    "환불은 언제까지 가능해요?",
    "고객센터 점심시간이 언제예요?",
    "문의 이메일 알려주세요.",
    "어떤 강의들을 제공하나요?",
    "채용 공고 있나요?",
]

for query in search_queries:
    result = run_workflow_until_search(query)
    print("=" * 60)
    print("query:", result["query"])
    print("label:", result["classification"]["label"])
    print("route target:", result["route"]["target"])
    print("search type:", result["search"]["type"])
    if result["search"]["matches"]:
        top_match = result["search"]["matches"][0]
        print("top passage:", top_match["passage"])
    else:
        print("top passage: 없음")

query: 환불은 언제까지 가능해요?
label: refund_policy
route target: refund_policy
search type: document_search
top passage: 환불은 고객센터 이메일 또는 대표 전화로 접수할 수 있으며 접수 후 영업일 기준 3일 이내 처리됩니다.
query: 고객센터 점심시간이 언제예요?
label: business_hours
route target: business_hours
search type: document_search
top passage: 고객센터 운영 시간은 평일 오전 10시부터 오후 6시까지입니다.
query: 문의 이메일 알려주세요.
label: contact_info
route target: contact_info
search type: document_search
top passage: 연락처
query: 어떤 강의들을 제공하나요?
label: other
route target: None
search type: no_search
top passage: 없음
query: 채용 공고 있나요?
label: other
route target: None
search type: no_search
top passage: 없음


## 8. Answer

`answer`는 `search`가 찾은 근거를 바탕으로 최종 사용자 응답을 만드는 단계다.

즉,
- `search` = 관련 근거 찾기
- `answer` = 그 근거를 바탕으로 사람에게 읽기 좋은 답변 만들기

여기서는 OpenAI Responses API를 사용해, 검색 결과를 근거로 최종 답변을 생성한다.

In [ ]:
# search가 찾은 근거를 바탕으로 최종 답변을 생성합니다.
def answer_step(query: str, search_result: dict) -> dict:
    matches = search_result.get("matches", [])

    # 관련도 점수가 0보다 큰 passage만 실제 근거로 사용합니다.
    useful_matches = [match for match in matches if match.get("score", 0) > 0]

    # 근거가 없으면 안전하게 fallback 답변을 반환합니다.
    if not useful_matches:
        return {
            "type": "fallback_answer",
            "answer": "지금 가진 문서만으로는 질문에 맞는 근거를 찾지 못했습니다. 다른 표현으로 다시 질문해 주세요.",
            "grounding": [],
        }

    # 모델에게 넘길 근거 문장을 구성합니다.
    grounding = [match["passage"] for match in useful_matches]
    context = "\n".join(f"- {passage}" for passage in grounding)
    # 근거 기반 답변 생성 프롬프트를 호출합니다.
    response = client.responses.create(
        model=MODEL_NAME,
        instructions=(
            "너는 주어진 근거만 사용해서 답하는 고객센터 어시스턴트다. "
            "근거에 없는 내용은 추측하지 말고, 짧고 명확하게 한국어로 답하라."
        ),
        input=(
            f"사용자 질문: {query}\n"
            f"근거:\n{context}"
        ),
    )

    # 답변 텍스트와 함께 어떤 근거를 썼는지도 반환합니다.
    return {
        "type": "llm_answer",
        "answer": response.output_text,
        "grounding": grounding,
    }

In [ ]:
# answer 단계가 실제로 어떤 답을 만드는지 확인합니다.
query = "환불은 언제까지 가능해요?"
state = run_workflow_until_search(query)
answer_result = answer_step(query, state["search"])

print("query:", query)
print("answer type:", answer_result["type"])
print()
print(answer_result["answer"])
print()
print("grounding:")
for item in answer_result["grounding"]:
    print("-", item)

## 9. Retry

`retry`는 첫 번째 시도가 만족스럽지 않을 때 다른 전략으로 다시 시도하는 단계다.

예를 들어 이런 경우 retry를 고려할 수 있다.
- `classify` 결과가 `other`인 경우
- `route`가 fallback 경로로 간 경우
- `search` 결과는 있지만 점수가 너무 낮아서 근거가 약한 경우

이번 노트북에서는 가장 단순한 retry 전략으로, 처음 경로가 실패하면 모든 문서를 다시 훑어보는 전역 검색을 사용한다.

In [ ]:
# 모든 문서를 대상으로 다시 검색하는 retry용 헬퍼 함수입니다.
def search_all_documents(query: str, top_k: int = 3) -> dict:
    ranked_matches = []

    # 각 문서를 하나씩 search_step에 넣어 전역 후보를 만듭니다.
    for target, content in docs.items():
        route_result = {
            "type": "document",
            "target": target,
            "content": content,
        }
        search_result = search_step(query, route_result, top_k=1)
        if search_result["matches"]:
            top_match = search_result["matches"][0]
            ranked_matches.append(
                {
                    "target": target,
                    "passage": top_match["passage"],
                    "score": top_match["score"],
                    "overlap_tokens": top_match["overlap_tokens"],
                }
            )

    # 전역 후보를 관련도 순으로 정렬합니다.
    ranked_matches.sort(key=lambda item: item["score"], reverse=True)

    return {
        "type": "global_document_search",
        "matches": ranked_matches[:top_k],
    }


# 현재 상태를 보고 retry가 필요한지 판단합니다.
def should_retry(state: dict) -> tuple[bool, list[str]]:
    reasons = []

    if state["classification"]["label"] == "other":
        reasons.append("classifier가 other를 반환했습니다.")

    if state["route"]["type"] == "fallback":
        reasons.append("route가 fallback 경로를 선택했습니다.")

    # 현재 search 결과의 최고 점수를 확인합니다.
    matches = state["search"].get("matches", [])
    best_score = matches[0]["score"] if matches else 0
    if best_score == 0:
        reasons.append("search 결과의 관련도 점수가 낮습니다.")

    return bool(reasons), reasons


# retry가 필요하면 더 넓은 범위의 검색 전략으로 다시 시도합니다.
def retry_step(state: dict) -> dict:
    retry_needed, reasons = should_retry(state)

    # 현재 결과가 충분하면 원래 search 결과를 그대로 유지합니다.
    if not retry_needed:
        return {
            "retried": False,
            "reason": "현재 결과로도 충분해서 retry를 수행하지 않습니다.",
            "search": state["search"],
        }

    # 실패 시 모든 문서를 대상으로 한 전역 검색으로 재시도합니다.
    global_search_result = search_all_documents(state["query"])
    useful_matches = [match for match in global_search_result["matches"] if match["score"] > 0]

    # 전역 검색에서 쓸 만한 근거를 찾으면 그 결과를 다음 단계로 넘깁니다.
    if useful_matches:
        return {
            "retried": True,
            "strategy": "global_document_search",
            "reason": " ".join(reasons),
            "search": {
                "type": "retry_search",
                "target": useful_matches[0]["target"],
                "matches": useful_matches,
            },
        }

    # 전역 검색에서도 근거를 못 찾으면 빈 결과로 마무리합니다.
    return {
        "retried": True,
        "strategy": "fallback_answer",
        "reason": " ".join(reasons),
        "search": {
            "type": "retry_search",
            "target": None,
            "matches": [],
        },
    }

In [ ]:
# retry가 실제로 언제 발동하는지 확인합니다.
query = "어떤 강의들을 제공하나요?"
state = run_workflow_until_search(query)
retry_result = retry_step(state)

print("query:", query)
print("initial label:", state["classification"]["label"])
print("initial route:", state["route"]["type"])
print("retry reason:", retry_result["reason"])
print("retried:", retry_result["retried"])
print("strategy:", retry_result.get("strategy"))
print()

for idx, match in enumerate(retry_result["search"]["matches"], start=1):
    print(f"[{idx}] target={match['target']} score={match['score']}")
    print(match["passage"])
    print()

## 10. classify + route + search + answer + retry 묶기

이제 모든 단계를 하나로 묶어보면 멀티스텝 workflow가 왜 필요한지 더 잘 보인다.

기본 흐름은 아래와 같다.

`query -> classify -> route -> search -> answer`

그리고 중간 결과가 약하면 `retry`가 개입해서 다시 검색 전략을 바꾸거나 fallback으로 넘어간다.

In [ ]:
# 전체 멀티스텝 workflow를 한 함수로 묶습니다.
def run_full_workflow(
    query: str,
    use_llm_classifier: bool = False,
) -> dict:
    # 먼저 classify -> route -> search까지 실행합니다.
    state = run_workflow_until_search(query, use_llm_classifier=use_llm_classifier)
    first_answer = answer_step(query, state["search"])
    retry_result = retry_step(state)

    # retry가 수행됐다면 retry 결과를, 아니면 기존 search 결과를 최종 근거로 사용합니다.
    final_search = retry_result["search"] if retry_result["retried"] else state["search"]
    final_answer = answer_step(query, final_search)

    # 단계별 결과를 모두 묶어서 반환합니다.
    return {
        "query": query,
        "classification": state["classification"],
        "route": state["route"],
        "initial_search": state["search"],
        "initial_answer": first_answer,
        "retry": retry_result,
        "final_answer": final_answer,
    }

In [ ]:
# 여러 질문으로 전체 workflow를 한 번에 점검합니다.
final_queries = [
    "환불은 언제까지 가능해요?",
    "고객센터 이메일 알려주세요.",
    "어떤 강의들을 제공하나요?",
    "채용 중인가요?",
]

for query in final_queries:
    result = run_full_workflow(query)
    print("=" * 60)
    print("query:", result["query"])
    print("label:", result["classification"]["label"])
    print("route:", result["route"]["type"], result["route"]["target"])
    print("retry used:", result["retry"]["retried"])
    print("final answer type:", result["final_answer"]["type"])
    print(result["final_answer"]["answer"])
    print()